In [0]:
# Databricks notebook source

from __future__ import annotations

import hashlib
import random
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import List, Dict, Tuple

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
@dataclass
class SampleConfig:
    seed: int = 42
    n_people: int = 250
    n_households: int = 110
    n_campaigns: int = 6
    min_events_per_person: int = 1
    max_events_per_person: int = 5
    provider_b_overlap_rate: float = 0.65
    provider_c_overlap_rate: float = 0.35
    catalog: str = "tim_dev"
    schema: str = "cleanroom"


cfg = SampleConfig(
    seed=42,
    n_people=250,
    n_households=110,
    n_campaigns=6,
    catalog="tim_dev",
    schema="cleanroom",
)

print(cfg)

In [0]:
def _sha256(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _pick(rng: random.Random, values: List[str]) -> str:
    return values[rng.randrange(len(values))]


def _random_ts(
    rng: random.Random,
    start_dt: datetime,
    end_dt: datetime,
) -> datetime:
    delta_seconds = int((end_dt - start_dt).total_seconds())
    offset = rng.randint(0, delta_seconds)
    return start_dt + timedelta(seconds=offset)


def _fqtn(config: SampleConfig, table_name: str) -> str:
    return f"{config.catalog}.{config.schema}.{table_name}"

In [0]:
def build_identity_spine_rows(config: SampleConfig) -> List[Dict[str, object]]:
    rng = random.Random(config.seed)

    first_names = [
        "James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael",
        "Linda", "David", "Elizabeth", "William", "Susan", "Richard", "Jessica",
        "Joseph", "Sarah", "Thomas", "Karen", "Charles", "Nancy", "Daniel",
        "Lisa", "Matthew", "Betty", "Anthony", "Margaret", "Mark", "Sandra",
    ]
    last_names = [
        "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller",
        "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez",
        "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin",
    ]
    email_domains = ["gmail.com", "yahoo.com", "outlook.com", "icloud.com", "proton.me"]
    states = ["CA", "NY", "TX", "FL", "IL", "WA", "CO", "GA", "NC", "MA"]
    providers = ["provider_a", "provider_b", "provider_c"]
    device_types = ["mobile", "connected_tv", "tablet", "desktop"]

    start_dt = datetime.now(timezone.utc) - timedelta(days=120)
    end_dt = datetime.now(timezone.utc)

    base_people: List[Dict[str, object]] = []
    for i in range(config.n_people):
        first = _pick(rng, first_names)
        last = _pick(rng, last_names)
        email = f"{first.lower()}.{last.lower()}{rng.randint(1, 999)}@{_pick(rng, email_domains)}"
        household_id = f"hh_{rng.randint(1, config.n_households):04d}"
        state = _pick(rng, states)

        base_people.append(
            {
                "person_id": f"person_{i + 1:04d}",
                "household_id": household_id,
                "email": email,
                "state": state,
                "first_seen_ts": _random_ts(rng, start_dt, end_dt - timedelta(days=30)),
                "last_seen_ts": _random_ts(rng, end_dt - timedelta(days=30), end_dt),
                "device_type": _pick(rng, device_types),
            }
        )

    rows: List[Dict[str, object]] = []

    for person in base_people:
        provider_membership = {"provider_a": True}
        provider_membership["provider_b"] = rng.random() < config.provider_b_overlap_rate
        provider_membership["provider_c"] = rng.random() < config.provider_c_overlap_rate

        for provider in providers:
            if not provider_membership.get(provider, False):
                continue

            provider_person_id = f"{provider}_{person['person_id']}"
            raw_device_id = f"{provider}:{person['person_id']}:{person['device_type']}"
            raw_email = person["email"]

            rows.append(
                {
                    "provider_id": provider,
                    "provider_person_id": provider_person_id,
                    "household_id": person["household_id"],
                    "hashed_email": _sha256(raw_email),
                    "hashed_device_id": _sha256(raw_device_id),
                    "state": person["state"],
                    "device_type": person["device_type"],
                    "first_seen_ts": person["first_seen_ts"],
                    "last_seen_ts": person["last_seen_ts"],
                    "match_confidence": round(rng.uniform(0.82, 0.99), 4),
                    "source_system": _pick(rng, ["crm", "site_events", "identity_graph"]),
                }
            )

    return rows

In [0]:
identity_rows = build_identity_spine_rows(cfg)

print(f"identity_rows: {len(identity_rows)}")
identity_rows[:5]

In [0]:
identity_schema = T.StructType(
    [
        T.StructField("provider_id", T.StringType(), False),
        T.StructField("provider_person_id", T.StringType(), False),
        T.StructField("household_id", T.StringType(), False),
        T.StructField("hashed_email", T.StringType(), False),
        T.StructField("hashed_device_id", T.StringType(), False),
        T.StructField("state", T.StringType(), True),
        T.StructField("device_type", T.StringType(), True),
        T.StructField("first_seen_ts", T.TimestampType(), True),
        T.StructField("last_seen_ts", T.TimestampType(), True),
        T.StructField("match_confidence", T.DoubleType(), True),
        T.StructField("source_system", T.StringType(), True),
    ]
)

identity_df = spark.createDataFrame(identity_rows, schema=identity_schema)

display(identity_df)

In [0]:
identity_df.groupBy("provider_id").count().orderBy("provider_id").show()
identity_df.groupBy("device_type").count().orderBy("device_type").show()
identity_df.groupBy("state").count().orderBy(F.desc("count")).show(10)

In [0]:
def build_ad_event_rows(
    config: SampleConfig,
    identity_rows: List[Dict[str, object]],
) -> List[Dict[str, object]]:
    rng = random.Random(config.seed + 1)

    campaign_defs: List[Tuple[str, str, str, str]] = [
        ("cmp_1001", "Max March Drama", "Warner Bros", "Streaming"),
        ("cmp_1002", "CNN Election Night", "CNN", "News"),
        ("cmp_1003", "HGTV Dream Home", "HGTV", "Lifestyle"),
        ("cmp_1004", "TNT Playoffs Push", "TNT", "Sports"),
        ("cmp_1005", "Discovery Adventure Week", "Discovery", "Entertainment"),
        ("cmp_1006", "Food Network Kitchen", "Food Network", "Food"),
    ]

    rows: List[Dict[str, object]] = []
    now = datetime.now(timezone.utc)

    for ident in identity_rows:
        n_events = rng.randint(config.min_events_per_person, config.max_events_per_person)

        for idx in range(n_events):
            campaign_id, campaign_name, brand, genre = campaign_defs[
                rng.randrange(min(config.n_campaigns, len(campaign_defs)))
            ]
            event_ts = _random_ts(rng, now - timedelta(days=45), now)

            rows.append(
                {
                    "event_id": _sha256(
                        f"{ident['provider_person_id']}|{campaign_id}|{event_ts.isoformat()}|{idx}"
                    ),
                    "provider_id": ident["provider_id"],
                    "provider_person_id": ident["provider_person_id"],
                    "hashed_email": ident["hashed_email"],
                    "hashed_device_id": ident["hashed_device_id"],
                    "household_id": ident["household_id"],
                    "campaign_id": campaign_id,
                    "campaign_name": campaign_name,
                    "advertiser_name": brand,
                    "genre": genre,
                    "device_type": ident["device_type"],
                    "event_ts": event_ts,
                    "event_date": event_ts.date(),
                    "impressions": rng.randint(1, 4),
                    "media_cost": round(rng.uniform(0.25, 8.5), 2),
                }
            )

    return rows

In [0]:
ad_event_rows = build_ad_event_rows(cfg, identity_rows)

print(f"ad_event_rows: {len(ad_event_rows)}")
ad_event_rows[:5]

In [0]:
ad_events_schema = T.StructType(
    [
        T.StructField("event_id", T.StringType(), False),
        T.StructField("provider_id", T.StringType(), False),
        T.StructField("provider_person_id", T.StringType(), False),
        T.StructField("hashed_email", T.StringType(), False),
        T.StructField("hashed_device_id", T.StringType(), False),
        T.StructField("household_id", T.StringType(), False),
        T.StructField("campaign_id", T.StringType(), False),
        T.StructField("campaign_name", T.StringType(), True),
        T.StructField("advertiser_name", T.StringType(), True),
        T.StructField("genre", T.StringType(), True),
        T.StructField("device_type", T.StringType(), True),
        T.StructField("event_ts", T.TimestampType(), True),
        T.StructField("event_date", T.DateType(), True),
        T.StructField("impressions", T.IntegerType(), True),
        T.StructField("media_cost", T.DoubleType(), True),
    ]
)

ad_events_df = spark.createDataFrame(ad_event_rows, schema=ad_events_schema)

display(ad_events_df)

In [0]:
ad_events_df.groupBy("campaign_id", "campaign_name").count().orderBy("campaign_id").show(truncate=False)
ad_events_df.groupBy("provider_id").agg(
    F.count("*").alias("events"),
    F.sum("impressions").alias("impressions"),
    F.round(F.sum("media_cost"), 2).alias("media_cost")
).orderBy("provider_id").show()

In [0]:
def build_overlap_results_df(identity_df):
    a = (
        identity_df.filter(F.col("provider_id") == "provider_a")
        .select("hashed_email")
        .distinct()
        .withColumnRenamed("hashed_email", "a_hashed_email")
    )

    b = (
        identity_df.filter(F.col("provider_id") == "provider_b")
        .select("hashed_email")
        .distinct()
        .withColumnRenamed("hashed_email", "hashed_email_b")
    )

    c = (
        identity_df.filter(F.col("provider_id") == "provider_c")
        .select("hashed_email")
        .distinct()
        .withColumnRenamed("hashed_email", "hashed_email_c")
    )

    def pair_stats(left_df, right_df, left_name: str, right_name: str, left_col: str, right_col: str):
        left_count_df = left_df.agg(F.count("*").alias("left_universe"))
        right_count_df = right_df.agg(F.count("*").alias("right_universe"))
        overlap_count_df = left_df.join(
            right_df,
            left_df[left_col] == right_df[right_col],
            "inner"
        ).agg(F.count("*").alias("overlap_count"))

        stats = left_count_df.crossJoin(right_count_df).crossJoin(overlap_count_df)

        return stats.select(
            F.lit(left_name).alias("left_provider_id"),
            F.lit(right_name).alias("right_provider_id"),
            F.col("left_universe"),
            F.col("right_universe"),
            F.col("overlap_count"),
            F.round(F.col("overlap_count") / F.col("left_universe"), 4).alias("left_overlap_rate"),
            F.round(F.col("overlap_count") / F.col("right_universe"), 4).alias("right_overlap_rate"),
            F.current_timestamp().alias("computed_ts"),
        )

    ab = pair_stats(a, b, "provider_a", "provider_b", "a_hashed_email", "hashed_email_b")
    ac = pair_stats(a, c, "provider_a", "provider_c", "a_hashed_email", "hashed_email_c")
    bc = pair_stats(b, c, "provider_b", "provider_c", "hashed_email_b", "hashed_email_c")

    return ab.unionByName(ac).unionByName(bc)

In [0]:
overlap_results_df = build_overlap_results_df(identity_df)

display(overlap_results_df)

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {cfg.catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {cfg.catalog}.{cfg.schema}")

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {_fqtn(cfg, "identity_spine")} (
        provider_id STRING,
        provider_person_id STRING,
        household_id STRING,
        hashed_email STRING,
        hashed_device_id STRING,
        state STRING,
        device_type STRING,
        first_seen_ts TIMESTAMP,
        last_seen_ts TIMESTAMP,
        match_confidence DOUBLE,
        source_system STRING
    )
    USING DELTA
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {_fqtn(cfg, "ad_events")} (
        event_id STRING,
        provider_id STRING,
        provider_person_id STRING,
        hashed_email STRING,
        hashed_device_id STRING,
        household_id STRING,
        campaign_id STRING,
        campaign_name STRING,
        advertiser_name STRING,
        genre STRING,
        device_type STRING,
        event_ts TIMESTAMP,
        event_date DATE,
        impressions INT,
        media_cost DOUBLE
    )
    USING DELTA
    PARTITIONED BY (event_date)
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {_fqtn(cfg, "overlap_results")} (
        left_provider_id STRING,
        right_provider_id STRING,
        left_universe BIGINT,
        right_universe BIGINT,
        overlap_count BIGINT,
        left_overlap_rate DOUBLE,
        right_overlap_rate DOUBLE,
        computed_ts TIMESTAMP
    )
    USING DELTA
    """
)

In [0]:
identity_df.write.format("delta").mode("overwrite").saveAsTable(_fqtn(cfg, "identity_spine"))
print(f"Wrote {_fqtn(cfg, 'identity_spine')}")

In [0]:
ad_events_df.write.format("delta").mode("overwrite").saveAsTable(_fqtn(cfg, "ad_events"))
print(f"Wrote {_fqtn(cfg, 'ad_events')}")

In [0]:
overlap_results_df.write.format("delta").mode("overwrite").saveAsTable(_fqtn(cfg, "overlap_results"))
print(f"Wrote {_fqtn(cfg, 'overlap_results')}")

In [0]:
for table_name in ["identity_spine", "ad_events", "overlap_results"]:
    fqtn = _fqtn(cfg, table_name)
    count = spark.table(fqtn).count()
    print(f"{fqtn}: {count:,} rows")

In [0]:
display(spark.table(_fqtn(cfg, "identity_spine")).orderBy("provider_id", "provider_person_id"))

In [0]:
display(spark.table(_fqtn(cfg, "ad_events")).orderBy(F.desc("event_ts")))

In [0]:
display(spark.table(_fqtn(cfg, "overlap_results")))